## 3. The task-specific cost metric

Everyone is infected. For a person with hospitalisation probability *h* (if unvaccinated) and vaccine effectiveness *VE* against hospitalisation:

| | Wants vaccine (y = 1) | Does not want it (y = 0) |
|---|---|---|
| **Sent (predict 1)** | 300 + h·(1 − VE)·18,500 (protected) | 300 + h·18,500 (wasted vaccine) |
| **Not sent (predict 0)** | 300 + h·18,500 (town pays cash, person unprotected) | h·18,500 |

**Total cost = sum over all people – lower is better.**

**Our assumptions (researched with generative AI, cross-checked across team members, as encouraged in the brief):**
- **Hospitalisation probability per infection by age**, from CDC burden estimates (hospitalisations ÷ symptomatic illnesses, H3N2-dominant season 2017-18): 18–44 ≈ 0.56 %, 45–54 ≈ 0.8 %, 55–64 ≈ 1.06 %, 65+ ≈ 9 %.
- **Chronic conditions** multiply risk by (1 + number of conditions) – a stated, simple assumption.
- **Vaccine effectiveness against hospitalisation:** VE = 0.40 (CDC: ≈ 41–61 %; H3N2 vaccines tend to the lower end).
- An **alternative set of literature estimates** from one of our team members (lower 65+ risk of 2.5 %, VE 0.22 for 65+) is tested in Section 9.

**Optimal decision rule.** Sending is worth it only if the expected hospital saving exceeds the expected vaccine waste:

$$\text{send if } P(\text{wants}) > \frac{300}{300 + 18{,}500 \cdot h \cdot VE}$$

So the threshold is **personal**: high (≈ 0.88) for young healthy people, low (≈ 0.1–0.3) for older or chronically ill people.

In [ ]:
VE = 0.40                                                    # vaccine effectiveness against hospitalisation
AGE_RISK = {"18": 0.0056, "35": 0.0056, "45": 0.008, "55": 0.0106, "65": 0.09}   # keyed by first 2 chars of age_group

def hosp_prob(df, age_risk=AGE_RISK, chronic_mult=1.0):
    """Per-person hospitalisation probability if unvaccinated: age risk x (1 + k * chronic conditions)."""
    base = df["age_group"].astype(str).str[:2].map(age_risk).fillna(0.0056)
    n_chronic = df["number_chronic_med_conditions"].fillna(0)
    return (base * (1 + chronic_mult * n_chronic)).clip(upper=1.0).to_numpy()

def cost_per_person(y_true, send, h, ve=VE):
    """Average $ cost per person for send(1)/don't-send(0) decisions. ve may be a number or an array."""
    y_true, send = np.asarray(y_true), np.asarray(send)
    cost = np.where(send == 1,
                    VACCINE_COST + np.where(y_true == 1, h * (1 - ve), h) * HOSPITAL_COST,   # vaccine sent
                    np.where(y_true == 1, VACCINE_COST, 0) + h * HOSPITAL_COST)            # not sent
    return cost.mean()

def personal_threshold(h, ve=VE):
    """Probability above which sending a vaccine reduces expected cost."""
    return VACCINE_COST / (VACCINE_COST + HOSPITAL_COST * h * ve)

h_train, h_test = hosp_prob(train), hosp_prob(test)
y = train[TARGET].astype(int).to_numpy()

# Sanity check: perfect decisions must beat random decisions
rng = np.random.default_rng(RANDOM_STATE)
assert cost_per_person(y, y, h_train) < cost_per_person(y, rng.integers(0, 2, len(y)), h_train)

# Personal thresholds for typical people
for age in ["18 - 34 Years", "55 - 64 Years", "65+ Years"]:
    for n in [0, 1]:
        h = hosp_prob(pd.DataFrame({"age_group": [age], "number_chronic_med_conditions": [n]}))[0]
        print(f"{age:15s} | {n} chronic condition(s) | h = {h:.4f} | send if P(wants) > {personal_threshold(h):.2f}")

In [ ]:
# Baselines: what would the town pay without a model? (cost per person, $)
n = len(y)
baselines = {
    "Send to nobody":    cost_per_person(y, np.zeros(n), h_train),
    "Send to everybody": cost_per_person(y, np.ones(n),  h_train),
    "Perfect knowledge (lower bound)": cost_per_person(y, y, h_train),
}
display(pd.Series(baselines).round(2).to_frame("Cost per person ($)"))

**Interpretation.** Sending a vaccine to everybody is already cheaper than sending to nobody, but perfect knowledge would be far cheaper still. The gap between "send to nobody" and "perfect knowledge" is the **maximum saving a model can achieve**.